In [0]:
# Install openpyxl for reading Excel files in pandas
%pip install openpyxl  

In [0]:
# Import required libraries for data processing and Spark DataFrame operations
import pandas as pd  # For reading Excel files and handling tabular data
import glob  # For file path pattern matching and retrieval
import re  # For regular expression operations, e.g., extracting dates from filenames
from pyspark.sql.functions import col, datediff, when, concat, coalesce, lit  # For Spark DataFrame column operations
from pyspark.sql.types import StructType, StructField, DateType, IntegerType  # For defining custom Spark DataFrame schemas

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})\.xlsx', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths):
    # Create an empty list to store DataFrames
    dfs = []

    for file in file_paths:
        # Extract the filename
        filename = file.split('/')[-1]
        
        # Read the Excel file
        df = pd.read_excel(file, dtype=str)
        
        # Extract date from filename for all files
        reporting_date = extract_date_from_filename(filename)
        df['Reporting Date'] = reporting_date
        
        dfs.append(df)
    return dfs

def read_write(excel_path, destination_path):
    # Define the schema
    custom_schema = StructType([
        StructField("Reporting Date", DateType(), nullable=False),
        StructField("Count", IntegerType(), nullable=False)
    ])

    # Read excel file
    pd_df = pd.read_excel(excel_path)

    # Create DataFrame using pandas df
    spark_df = spark.createDataFrame(pd_df,schema=custom_schema)

    # Write the result to destination in parquet
    spark_df.write.mode("overwrite").parquet(destination_path)
    print(f"Data has been successfully processed and written at {destination_path}")

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/advances'
destination_path_direct_debit = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/direct_debit'
destination_path_supplier_recs = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/supplier_recs'

companycode_path = source_path+'input_files/Company Code.xlsx'
filters_advances_path = source_path+'input_files/Filters - Advances.xlsx'
staff_mapping_path = source_path+'input_files/Staff Mapping.xlsx'
supplier_mapping_path = source_path+'input_files/Supplier Mapping.xlsx'

advances_file_paths = source_path + 'advances/Advances*.xlsx'
direct_debit_path = source_path + "direct_debit/Direct Debits (in Bank Ledger).xlsx"
supplier_recs_path = source_path + "supplier_recs/Number of Supplier Recs completed.xlsx"

In [0]:
# Reading Static Tables - CompanyCode, Filter Advances

# Reading CompanyCode and convert the Pandas DataFrame to a Spark DataFrame
companycode_df = pd.read_excel(companycode_path, dtype=str)
companycode = spark.createDataFrame(companycode_df)

# Reading Filter Advances and convert the Pandas DataFrame to a Spark DataFrame
filters_advances_df = pd.read_excel(filters_advances_path, dtype=str)
filters_advances = spark.createDataFrame(filters_advances_df)

# Reading Dynamic Tables - Staff Mapping, Supplier Mapping

# Reading Staff Mapping 
staff_mapping_df = pd.read_excel(staff_mapping_path, dtype=str)
staff_mapping = spark.createDataFrame(staff_mapping_df)

# Reading Supplier Mapping 
supplier_mapping_df = pd.read_excel(supplier_mapping_path, dtype=str)
supplier_mapping = spark.createDataFrame(supplier_mapping_df)

In [0]:
# Define the path to your Excel files
file_paths = glob.glob(advances_file_paths)

# Add Reporting Date Column
dfs = add_reporting_date(file_paths)

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Convert the Pandas DataFrame to a Spark DataFrame
advances_input = spark.createDataFrame(combined_df)

# Add the new columns
advances_input = advances_input.withColumn(
    "Difference", datediff(col("Reporting Date"), col("Net due date"))
).withColumn(
    "Ageing",
    when(col("Difference").isNull(), 'Unknown') \
    .when(col("Difference") <= 30, '0-30 Days') \
    .when((col("Difference") > 30) & (col("Difference") <= 60), '31-60 Days') \
    .when((col("Difference") > 60) & (col("Difference") <= 90), '61-90 Days') \
    .when((col("Difference") > 90) & (col("Difference") <= 365), '91-365 Days') \
    .otherwise('More than 1 year')
)

# Join with companycode to get Company_Name
advances_input = advances_input.join(companycode, advances_input["Company Code"] == companycode["Company Code"], "left").drop(companycode["Company Code"])

# Join with filters_advances to get GL_Flag
advances_input = advances_input.join(filters_advances, advances_input["G/L Account"] == filters_advances["GL ID"], "left").drop("GL ID", "GL Description")

advances_input = advances_input.withColumn(
    "Conc CC Vendor", concat(col("Company Code"), col("Vendor"))
)

# Drop duplicates to keep only one row per CC_Code_Vendor
supplier_mapping_unique = supplier_mapping.dropDuplicates(["CC Code+Vendor"])

# Join with supplier_mapping_unique to get Vendor Mapping
advances_input = advances_input.join(supplier_mapping_unique, advances_input["Conc CC Vendor"] == supplier_mapping_unique["CC Code+Vendor"], "left") \
    .drop(supplier_mapping_unique.Vendor).drop(supplier_mapping_unique["Company Code"]) \
    .drop("CC Code+Vendor", "Team Lead","Vendor type","Category","Vendor Status") \
    .withColumnRenamed("Manager", "Vendor Mapping") \
    .withColumn("Vendor Mapping", coalesce(col("Vendor Mapping"), lit('Unassigned')))

# Drop duplicates to keep only one row per Employee Number
staff_mapping_unique = staff_mapping.dropDuplicates(["Employee Number"])

# Join with staff_mapping to get staff
advances_input = advances_input.join(staff_mapping_unique, advances_input["User Name"] == staff_mapping_unique["Employee Number"], "left") \
    .drop("Employee Number", "Team Lead", "Line Manager","Status","Scope") \
    .withColumnRenamed("Employee Full Name", "Staff Mapping") \
    .withColumn("Staff Mapping", coalesce(col("Staff Mapping"), lit('Unassigned')))

# Write the data to the destination
advances_input.write.mode("overwrite").parquet(destination_path)
print(f"Data has been successfully processed and written at {destination_path}")

In [0]:
# Process direct debit
read_write(direct_debit_path, destination_path_direct_debit)

# Process supplier recs
read_write(supplier_recs_path, destination_path_supplier_recs)